<a href="https://colab.research.google.com/github/AENDYSTUDIO/Colab-notebooks/blob/main/NEUROFLAC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os
import shutil
import asyncio
import sqlite3
from datetime import datetime
import re

try:
    import telethon
except ImportError:
    !pip install telethon nest_asyncio
    import telethon

try:
    import mutagen
except ImportError:
    !pip install mutagen
    import mutagen

import nest_asyncio
from telethon import TelegramClient
from telethon.errors import FloodWaitError
from mutagen.flac import FLAC, Picture
from mutagen.mp3 import MP3
from mutagen.id3 import ID3, APIC, TIT2, TPE1, TCON, error

# Авторизация в Google Drive
drive.mount('/content/drive')

# --- НАСТРОЙКИ ---ы
API_ID = 13110281
API_HASH = 'c571e4ada0d7e42a9f9a5daba40a71be'
CHANNEL_USERNAME = 'neuroflac'

DRIVE_TARGET_DIR = '/content/drive/MyDrive/NEUROFLAC_ARCHIVE'
SESSION_PATH = '/content/drive/MyDrive/NEUROFLAC_ARCHIVE/colab_tg_session'
LOCAL_TEMP_DIR = '/content/temp_music'
DB_PATH = os.path.join(DRIVE_TARGET_DIR, 'neuroflac_archive.db')

os.makedirs(DRIVE_TARGET_DIR, exist_ok=True)
os.makedirs(LOCAL_TEMP_DIR, exist_ok=True)

def clean_text(text):
    if not text: return text
    text = re.sub(r'\*\*|__|`|\\', '', text)
    return text.strip()

def init_db():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS files (
            message_id INTEGER PRIMARY KEY,
            original_name TEXT,
            cleaned_file_name TEXT,
            genre TEXT,
            drive_file_path TEXT,
            file_size INTEGER,
            downloaded_at TEXT,
            artist TEXT,
            title TEXT
        )
    ''')
    conn.commit()
    conn.close()

def apply_metadata(file_path, artist, title, genre, cover_path=None):
    try:
        if file_path.endswith('.flac'):
            audio = FLAC(file_path)
            audio['artist'] = artist
            audio['title'] = title
            audio['genre'] = genre
            if cover_path and os.path.exists(cover_path):
                image = Picture()
                with open(cover_path, 'rb') as f:
                    image.data = f.read()
                image.type = 3 # Front Cover
                image.mime = 'image/jpeg'
                audio.add_picture(image)
            audio.save()
        elif file_path.endswith('.mp3'):
            try:
                audio = ID3(file_path)
            except error:
                audio = ID3()
            audio.add(TPE1(encoding=3, text=artist))
            audio.add(TIT2(encoding=3, text=title))
            audio.add(TCON(encoding=3, text=genre))
            if cover_path and os.path.exists(cover_path):
                with open(cover_path, 'rb') as f:
                    audio.add(APIC(encoding=3, mime='image/jpeg', type=3, desc='Cover', data=f.read()))
            audio.save(file_path)
    except Exception as e: print(f"Ошибка тегов: {e}")

client = TelegramClient(SESSION_PATH, API_ID, API_HASH)

async def main():
    print("Запуск синхронизации (Метаданные + Обложки)...")
    init_db()
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    async for message in client.iter_messages(CHANNEL_USERNAME):
        if message.audio:
            artist = getattr(message.audio, 'performer', "") or "Unknown Artist"
            title = getattr(message.audio, 'title', "") or "Unknown Title"

            if (artist == "Unknown Artist" or title == "Unknown Title") and message.text:
                lines = message.text.split('\n')
                first_line = re.sub(r'^(Name|Artist|Имя|Артист):\s*', '', lines[0].strip(), flags=re.IGNORECASE)
                match = re.match(r'(.+?)\s*[-–]\s*(.+)', first_line)
                if match:
                    artist = match.group(1).strip() if artist == "Unknown Artist" else artist
                    title = match.group(2).strip() if title == "Unknown Title" else title

            artist = clean_text(artist)
            title = clean_text(title)
            tags = re.findall(r'#(\w+)', message.text) if message.text else []
            genre_str = ', '.join(tags) if tags else "Unsorted"

            cursor.execute('SELECT message_id FROM files WHERE message_id = ?', (message.id,))
            if cursor.fetchone(): continue

            cover_file = None
            has_cover = False
            if message.photo:
                cover_file = os.path.join(LOCAL_TEMP_DIR, f"cover_{message.id}.jpg")
                await client.download_media(message, cover_file)
                has_cover = True

            cover_status = "🖼️ Обложка OK" if has_cover else "❌ Без обложки"
            print(f"-> {artist} - {title} [{genre_str[:20]}...] | {cover_status}")

            attr = message.audio.attributes[0]
            original_name = getattr(attr, 'file_name', f"{message.id}.flac")
            safe_name = f"{message.id}_" + "".join([c for c in original_name if c.isalnum() or c in ' ._-']).strip()

            local_path = os.path.join(LOCAL_TEMP_DIR, safe_name)
            drive_path = os.path.join(DRIVE_TARGET_DIR, (tags[0] if tags else "Unsorted"), safe_name)
            os.makedirs(os.path.dirname(drive_path), exist_ok=True)

            try:
                await client.download_media(message.audio, local_path)
                apply_metadata(local_path, artist, title, genre_str, cover_file)
                shutil.move(local_path, drive_path)
                cursor.execute('INSERT INTO files VALUES (?,?,?,?,?,?,?,?,?)',
                               (message.id, original_name, safe_name, genre_str, drive_path, message.audio.size, datetime.now().isoformat(), artist, title))
                conn.commit()
            except Exception as e: print(f"Ошибка загрузки {message.id}: {e}")
            finally:
                if cover_file and os.path.exists(cover_file): os.remove(cover_file)

    conn.close()
    print("Готово.")

nest_asyncio.apply()
async with client:
    await client.start()
    await main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Запуск синхронизации (Метаданные + Обложки)...
-> Unknown Artist - Unknown Title [Unsorted...] | ❌ Без обложки
-> Unknown Artist - Unknown Title [Unsorted...] | ❌ Без обложки
-> Unknown Artist - Unknown Title [Unsorted...] | ❌ Без обложки
-> Unknown Artist - Unknown Title [Unsorted...] | ❌ Без обложки
-> Unknown Artist - Unknown Title [Unsorted...] | ❌ Без обложки
-> Unknown Artist - Unknown Title [Unsorted...] | ❌ Без обложки
-> Unknown Artist - Unknown Title [Unsorted...] | ❌ Без обложки
-> Unknown Artist - Unknown Title [Unsorted...] | ❌ Без обложки
-> Unknown Artist - Unknown Title [Unsorted...] | ❌ Без обложки
-> Unknown Artist - Unknown Title [Unsorted...] | ❌ Без обложки
-> Unknown Artist - Unknown Title [Unsorted...] | ❌ Без обложки
-> Unknown Artist - Unknown Title [Unsorted...] | ❌ Без обложки
-> Unknown Artist - Unknown Title [Unsorted...] | ❌ Без 

In [ ]:
import asyncio
import nest_asyncio

# Применяем nest_asyncio для корректной работы в Colab
nest_asyncio.apply()

async def verify_new_logic():
    print("--- Проверка обновленной логики (Метаданные + Очистка + Обложки) ---")
    # Мы запускаем main() из первой ячейки, так как она уже содержит все правки
    async with client:
        # client.start() подхватит существующую сессию
        await client.start()
        await main()

await verify_new_logic()

--- Проверка обновленной логики (Метаданные + Очистка + Обложки) ---
Запуск скачивания (User API)...
Скачиваю: Unknown Artist - Unknown Title (Unsorted)
Скачиваю: Unknown Artist - Unknown Title (Unsorted)
Скачиваю: Unknown Artist - Unknown Title (Unsorted)
Скачиваю: Unknown Artist - Unknown Title (Unsorted)


In [ ]:
async def test_single_message_parsing():
    async with client:
        # Получаем последнее сообщение с аудио из канала
        async for message in client.iter_messages(CHANNEL_USERNAME, limit=10):
            if message.audio:
                print(f"--- Тестовый парсинг сообщения ID: {message.id} ---")

                # Исходные данные
                raw_text = message.text if message.text else "[Нет текста]"
                print(f"Текст сообщения:\n{raw_text}\n")

                # Логика извлечения
                artist = getattr(message.audio, 'performer', "") or "Unknown Artist"
                title = getattr(message.audio, 'title', "") or "Unknown Title"

                if (artist == "Unknown Artist" or title == "Unknown Title") and message.text:
                    lines = message.text.split('\n')
                    first_line = lines[0].strip() if lines else ''
                    match = re.match(r'(.+?)\s*[-–]\s*(.+)', first_line)
                    if match:
                        artist = match.group(1).strip() if artist == "Unknown Artist" else artist
                        title = match.group(2).strip() if title == "Unknown Title" else title

                tags = re.findall(r'#(\w+)', message.text) if message.text else []
                genre = ', '.join(tags) if tags else "Unsorted"

                print(f"РЕЗУЛЬТАТ:")
                print(f"Исполнитель: {artist}")
                print(f"Название:    {title}")
                print(f"Жанры (теги): {genre}")
                print(f"Папка:        {tags[0] if tags else 'Unsorted'}")
                break

# Запуск теста
import asyncio
await test_single_message_parsing()

NameError: name 'client' is not defined

In [ ]:
import re

def clean_text(text):
    if not text: return text
    # Удаляем жирный шрифт (**), курсив (__), код (`)
    text = re.sub(r'\*\*|__|`|\\', '', text)
    # Удаляем лишние пробелы
    return text.strip()

async def test_parsing_with_format_cleaned():
    async with client:
        print("Поиск сообщения с форматом 'Artist - Title' (с очисткой)...")
        found = False
        async for message in client.iter_messages(CHANNEL_USERNAME, limit=50):
            if message.text and (' - ' in message.text or ' – ' in message.text):
                lines = message.text.split('\n')
                # Берем первую строку и очищаем её от префиксов типа 'Name:' или 'Artist:'
                first_line = lines[0].strip()
                first_line = re.sub(r'^(Name|Artist|Имя|Артист):\s*', '', first_line, flags=re.IGNORECASE)

                match = re.match(r'(.+?)\s*[-–]\s*(.+)', first_line)
                if match:
                    # Применяем финальную очистку к результатам
                    artist = clean_text(match.group(1))
                    title = clean_text(match.group(2))
                    tags = re.findall(r'#(\w+)', message.text)
                    genre = ', '.join(tags) if tags else "Unsorted"

                    print(f"--- ТЕСТ С ОЧИСТКОЙ (ID: {message.id}) ---")
                    print(f"Оригинал: {lines[0].strip()}")
                    print(f"Очищено:")
                    print(f"  Исполнитель: {artist}")
                    print(f"  Название:    {title}")
                    print(f"  Теги:        {genre}")
                    found = True
                    break

        if not found:
            print("Подходящих сообщений не найдено.")

import asyncio
await test_parsing_with_format_cleaned()

NameError: name 'client' is not defined